# 05_06 Position-keyed event lift: fixing the Pseudo 890 pre-holiday underforecasts

This notebook documents one small, attributable feature-engineering iteration on the
two-stage LightGBM model. It adds **one mechanism** (three feature columns) to the
feature builder, refits **only the two-stage model**, and compares the result against
the previous run, which was backed up before refitting.

The target problem comes from `05_04`: the three worst Pseudo 890 weeks
(2026-05-18, 2026-04-27, 2026-03-30) were all **level underforecasts**
(weekly forecast/actual ratios 0.757, 0.777, 0.806), and in each week the error was
concentrated on the last trading days before a public-holiday closure block.

**Headline result:** overall pooled bias improved from −1.97% to −0.96%
(Pseudo 890: −2.21% → −1.21%) with overall WAPE flat (+0.11 pp). Two of the three
bad weeks improved on exactly the targeted pre-closure days; the Erster Mai peak
day did not, for a training-coverage reason quantified in the conclusions.

## Step-by-step: what was done and why

**Step 1 — Reproduce the diagnosis.** From `05_04`: the three largest
absolute-error Pseudo 890 weeks all sit around public holidays
(Karfreitag/Easter, Erster Mai, Pfingstmontag). Within each week the miss is
concentrated: 2026-04-30 (Thursday before the May 1 closure) alone carried 52.9%
of its week's error at a forecast/actual ratio of 0.597; 2026-05-22/23 (Friday and
Saturday before the Pentecost Sunday+Monday closure) carried 68.8% at ratios
0.656/0.653; the Saturday between Karfreitag and the Easter Sunday+Monday closure
carried 35.1% at 0.773. The two-stage decomposition showed the **positive-quantity
stage** carried most of the miss (positive-mean ratios 0.76–0.83) with a smaller
occurrence shortfall (−0.7 to −2.6 pp).

**Step 2 — Audit the event-lift features for bugs.** The event-lift implementation in
`src/models/lightgbm/features/builder.py` was reviewed end to end: leakage windows
(`feature_date < origin` everywhere), the 24-active-row maturity gate on both event
and baseline observations, the weekday-keyed article baseline with its
sourcing-group/category fallback, the closure-block-length cell key, and the
occurrence/quantity split that mirrors the two-stage objective. **No computational
bug was found.** The values on the bad days were exactly what the design specifies.

**Step 3 — Identify the structural gap.** What the design specifies is too coarse:
the pooled lift is keyed only by `(event, closure-block length)`, so **every day of a
±3-day event window receives the same number**. For the Erster Mai window the model
saw occurrence/quantity lifts of 1.170/1.437 on Tuesday *and* on the Thursday
before the closed Friday — but the real lift is concentrated almost entirely on that
Thursday. Pooling over the window dilutes the peak into the shoulders. The
row-varying alternative, `event_lift_series`, is a single prior-year observation per
series (median 0.0 on these days — most series sold nothing at the anchor date), so
the trees cannot recover the peak from it either.

**Step 4 — Verify the position signal exists and is stable.** Before writing any
code, the position-resolved lift was measured directly from raw history (table
below): the day before Erster Mai lifts Pseudo 890 demand ~3–4× in **every** year
2024–2026, the day before Christi Himmelfahrt ~2.4–2.8×, the Saturday before the
Pentecost closure ~1.3–1.7×, while days 3–4 away from the event sit near 1.0. The
signal is large, position-specific, and repeats across years — ideal for a pooled
historical feature.

**Step 5 — Add one mechanism, three columns.** The pooled-lift cell key was extended
with the target's **signed calendar-day offset to the event**
(`days_to_nearest_event`, already computed in the calendar):

- `event_position_lift_occurrence` — occurrence stage only,
- `event_position_lift_quantity` — quantity stage only,
- `event_position_lift_total` — their product, for the direct (single-stage) models.

Everything else is reused unchanged: the same maturity gates, the same weekday-keyed
baselines, the same strict `< origin` history. A position cell holds exactly one
date per historical event occurrence, so the support gate is **≥ 2 distinct dates**
(= two different years), which averages out single-day shocks such as weather. The
window-level features were left untouched (no clear bug → do not change), so the
model keeps the smoother window estimate as a fallback where position cells are thin.

**Step 6 — Rebuild and refit.** `FEATURE_BUILDER_VERSION` was bumped so the feature
store rebuilt all 72 origin partitions, and **only the two-stage model** was refit
(same hyperparameters, same expanding-window protocol, 5 refits over 20 evaluation
origins). The previous run's artifacts were copied to
`reports/results/backup_pre_position_lift/` and serve as the "old" side of every
comparison below.

## Why the previous underforecasts happened

Putting the audit and the evidence together, the underforecasts on the bad
origins had one dominant cause and two amplifiers:

1. **Stock-up demand before holiday closure blocks is position-specific, but no
   feature carried the position-specific magnitude.** Customers concentrate
   purchases on the last one or two trading days before a store closure that
   contains a public holiday (and Germany's Sunday-closure rule regularly turns a
   Monday holiday into a two-day block). The only features that could quantify the
   spike were the window-pooled lifts — one constant for all ±3 days, roughly the
   window *average* (~1.4–1.8 total) where the peak day needs ~3–4 — and the
   noisy single-anchor `event_lift_series` (median 0). Calendar categoricals
   (`event_name`, `days_to_nearest_event`, `closed_days_next_*`) mark *where* the
   spike is but not *how large* it is; the trees would have to reconstruct the
   magnitude from one or two training examples per event and year, which gradient
   boosting with pooled, regularized leaves does not do reliably.
2. **The quantity stage bears the miss.** The Gamma booster is trained on all
   positive rows, where pre-closure days are a tiny minority; without an explicit
   magnitude feature it predicts near the ordinary conditional mean, giving the
   observed positive-mean ratios of 0.76–0.83 on those weeks.
3. **The same dilution slightly inflates shoulder days** (e.g. Tuesday 2026-04-28
   at ratio 1.053 while Thursday sat at 0.597), so the weekly bias hid part of the
   daily error — visible only at daily grain.

In [1]:
from pathlib import Path
import json
import sys

import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.models.benchmark import load_benchmark_design
from src.models.lightgbm import TWO_STAGE_MODEL_NAME
from src.models.lightgbm.features.builder import _holiday_calendar
from src.models.lightgbm.features.store import FEATURE_BUILDER_VERSION
from src.models.results import result_path

pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 120)

SOURCE_GROUP = 'Pseudo'
CATEGORY_ID = 890
BAD_ORIGINS = ['2026-03-30', '2026-04-27', '2026-05-18']

design = load_benchmark_design()
new_forecast_path = result_path(TWO_STAGE_MODEL_NAME, design)
old_forecast_path = ROOT / 'reports' / 'results' / 'backup_pre_position_lift' / 'forecasts_two_stage.csv'

con = duckdb.connect()
con.execute('PRAGMA threads=4')
for label, path in [('old', old_forecast_path), ('new', new_forecast_path)]:
    con.execute(
        f'''
        CREATE OR REPLACE TEMP TABLE forecasts_{label} AS
        SELECT
            ARTIKEL_ID::BIGINT AS ARTIKEL_ID, MARKT_ID::BIGINT AS MARKT_ID,
            sourcing_group, category_id::INTEGER AS category_id,
            CAST(origin AS DATE) AS origin, CAST(period AS DATE) AS period,
            actual::DOUBLE AS actual, forecast::DOUBLE AS forecast,
            occurrence_probability::DOUBLE AS occurrence_probability,
            positive_quantity_forecast::DOUBLE AS positive_quantity_forecast
        FROM read_csv_auto(?)
        WHERE is_active
        ''',
        [str(path)],
    )
alignment = con.execute('''
    SELECT
        (SELECT COUNT(*) FROM forecasts_old) AS old_rows,
        (SELECT COUNT(*) FROM forecasts_new) AS new_rows,
        (SELECT COUNT(*) FROM forecasts_old o
         INNER JOIN forecasts_new n USING (ARTIKEL_ID, MARKT_ID, origin, period)
         WHERE o.actual <> n.actual) AS actual_mismatches
''').fetchdf()
display(alignment)
assert alignment.loc[0, 'old_rows'] == alignment.loc[0, 'new_rows']
assert alignment.loc[0, 'actual_mismatches'] == 0

,old_rows,new_rows,actual_mismatches
0,2502829,2502829,0


## Evidence: the position-resolved lift in the raw history

Each value is Pseudo 890 total demand on the given day relative to a same-weekday
baseline (median of the same weekday in surrounding non-event weeks). Offset is in
calendar days relative to the event day (−1 = day before). This is the pattern the
old features could not express: huge, **repeatable** lift at offsets −1/−2 and
nothing at the window edges.

In [2]:
transactions = str(ROOT / 'data' / 'processed' / 'transactions' / '*.parquet')
daily = duckdb.sql(f'''
    SELECT CAST(DATE AS DATE) AS period,
           SUM(CASE WHEN is_active THEN COALESCE(ABVERKAUFTE_MENGE_KG, 0) ELSE 0 END) AS demand
    FROM read_parquet('{transactions}')
    WHERE is_pseudo AND WGR_ID = {CATEGORY_ID}
    GROUP BY 1
''').fetchdf()
daily['period'] = pd.to_datetime(daily.period)
daily = daily.set_index('period')

EVENT_OCCURRENCES = {
    'Karfreitag': ['2024-03-29', '2025-04-18', '2026-04-03'],
    'Erster Mai': ['2024-05-01', '2025-05-01', '2026-05-01'],
    'Christi Himmelfahrt': ['2024-05-09', '2025-05-29', '2026-05-14'],
    'Pfingstmontag': ['2024-05-20', '2025-06-09', '2026-05-25'],
}
all_event_days = [pd.Timestamp(day) for days in EVENT_OCCURRENCES.values() for day in days]

def same_weekday_baseline(day):
    candidates = []
    for week in [-6, -5, -4, -3, 3, 4, 5, 6]:
        candidate = day + pd.Timedelta(days=7 * week)
        if candidate not in daily.index or daily.loc[candidate, 'demand'] == 0:
            continue
        if any(abs((candidate - event).days) <= 3 for event in all_event_days):
            continue
        candidates.append(daily.loc[candidate, 'demand'])
    return np.median(candidates) if len(candidates) >= 3 else np.nan

records = []
for event, occurrences in EVENT_OCCURRENCES.items():
    for occurrence in occurrences:
        event_day = pd.Timestamp(occurrence)
        for offset in range(-3, 4):
            day = event_day + pd.Timedelta(days=offset)
            if day not in daily.index or daily.loc[day, 'demand'] == 0:
                continue
            baseline = same_weekday_baseline(day)
            if pd.notna(baseline):
                records.append({
                    'event': event, 'year': event_day.year, 'offset': offset,
                    'lift': daily.loc[day, 'demand'] / baseline,
                })
position_lift_history = (
    pd.DataFrame(records)
    .pivot_table(index=['event', 'offset'], columns='year', values='lift')
    .round(2)
)
display(position_lift_history)

year                        2024  2025  2026
event               offset                  
Christi Himmelfahrt -3      0.99  0.80  0.78
                    -2      1.44  1.10  1.15
                    -1      2.82  2.62  2.45
                     1      1.13  1.01  0.93
                     2      1.12  0.92  0.78
Erster Mai          -3       NaN  1.09  0.96
                    -2      1.60  1.78  1.67
                    -1      3.48  4.06  3.01
                     1      1.16  0.97  1.22
                     2      1.01  0.90   NaN
                     3      1.01   NaN  0.94
Karfreitag          -3      1.82  1.20  1.70
                    -2      2.37  1.62  2.04
                    -1      2.55  1.73  2.17
                     1      1.77  1.22  1.57
Pfingstmontag       -3      1.30  1.23  1.70
                    -2      1.45  1.34  1.73
                     1      1.09  1.11  0.98
                     2      1.01  1.00  0.92
                     3      0.99  1.06  0.97

## The new feature on the three bad weeks

Values read from the rebuilt feature partitions. The window-level pooled lifts
(unchanged) are constant across each event window; the new position-keyed lifts
resolve the peak days. Missing position values are expected where a position cell
has fewer than two historical dates (for example an offset that fell on a Sunday in
one of the historical years) — there the model falls back to the window features.

In [3]:
FEATURE_STORE_ROOT = ROOT / 'data' / 'processed' / 'model_features'

def current_feature_generation():
    matches = []
    for manifest_path in FEATURE_STORE_ROOT.rglob('manifest.json'):
        try:
            manifest = json.loads(manifest_path.read_text())
        except (OSError, json.JSONDecodeError):
            continue
        signature = manifest.get('signature', {})
        if signature.get('feature_builder_version') == FEATURE_BUILDER_VERSION:
            matches.append((manifest_path.stat().st_mtime, manifest_path.parent))
    if not matches:
        raise FileNotFoundError(
            f'No feature generation with builder version {FEATURE_BUILDER_VERSION}'
        )
    return max(matches)[1]

feature_generation = current_feature_generation()
print(f'Feature generation: {feature_generation}')
bad_partitions = [
    str(feature_generation / f'origin={origin}' / 'features.parquet')
    for origin in BAD_ORIGINS
]
bad_day_features = con.execute(
    f'''
    SELECT
        CAST(origin AS DATE) AS origin, CAST(period AS DATE) AS period,
        ANY_VALUE(event_name) AS event_name,
        ANY_VALUE(days_to_nearest_event) AS days_to_event,
        ANY_VALUE(event_lift_pooled_occurrence) AS window_occurrence,
        ANY_VALUE(event_lift_pooled_quantity) AS window_quantity,
        ANY_VALUE(event_position_lift_occurrence) AS position_occurrence,
        ANY_VALUE(event_position_lift_quantity) AS position_quantity,
        ANY_VALUE(event_position_lift_total) AS position_total
    FROM read_parquet(?, hive_partitioning=false)
    WHERE sourcing_group = ? AND category_id = ? AND is_active
    GROUP BY origin, period
    ORDER BY origin, period
    ''',
    [bad_partitions, SOURCE_GROUP, CATEGORY_ID],
).fetchdf()
display(bad_day_features.round(3))

Feature generation: /Users/vlada/UNI/SoSe2026/ba/ba_code/data/processed/model_features/lightgbm_daily/v1/aa0326e586fe432e2293


,origin,period,event_name,days_to_event,window_occurrence,window_quantity,position_occurrence,position_quantity,position_total
0,2026-03-30,2026-03-30,none,4,NaN,NaN,NaN,NaN,NaN
1,2026-03-30,2026-03-31,Karfreitag,3,1.238,1.492,1.163,1.305,1.518
2,2026-03-30,2026-04-01,Karfreitag,2,1.238,1.492,1.289,1.617,2.084
3,2026-03-30,2026-04-02,Karfreitag,1,1.238,1.492,1.379,1.659,2.288
4,2026-03-30,2026-04-04,Karfreitag,-1,1.238,1.492,1.144,1.400,1.601
5,2026-04-27,2026-04-27,none,4,NaN,NaN,NaN,NaN,NaN
6,2026-04-27,2026-04-28,Erster Mai,3,1.170,1.437,NaN,NaN,NaN
7,2026-04-27,2026-04-29,Erster Mai,2,1.170,1.437,1.253,1.523,1.907
8,2026-04-27,2026-04-30,Erster Mai,1,1.170,1.437,1.534,2.819,4.324
9,2026-04-27,2026-05-02,Erster Mai,-1,1.170,1.437,1.064,1.093,1.164


## Pseudo 890: old versus new, every evaluation origin

WAPE and ratio follow `05_04`: forecasts and actuals are pooled across stores to
article-days first; WAPE is the sum of absolute article-day errors over actual
kilograms, ratio is forecast over actual kilograms. The three investigated weeks are
marked. A ratio below 1 is an underforecast.

In [4]:
def origin_metrics(label):
    return con.execute(f'''
        WITH article_day AS (
            SELECT ARTIKEL_ID, origin, period,
                   SUM(actual) AS actual, SUM(forecast) AS forecast
            FROM forecasts_{label}
            WHERE sourcing_group = '{SOURCE_GROUP}' AND category_id = {CATEGORY_ID}
            GROUP BY 1, 2, 3
        )
        SELECT origin,
               SUM(actual) AS actual_kg,
               SUM(ABS(forecast - actual)) AS absolute_error_kg,
               SUM(ABS(forecast - actual)) / NULLIF(SUM(actual), 0) AS wape,
               SUM(forecast) / NULLIF(SUM(actual), 0) AS ratio
        FROM article_day
        GROUP BY origin ORDER BY origin
    ''').fetchdf()

pseudo_890 = origin_metrics('old').merge(
    origin_metrics('new'), on='origin', suffixes=('_old', '_new'), validate='one_to_one'
)
pseudo_890 = pseudo_890.drop(columns=['actual_kg_new']).rename(columns={'actual_kg_old': 'actual_kg'})
pseudo_890['wape_change_pp'] = 100 * (pseudo_890.wape_new - pseudo_890.wape_old)
pseudo_890['investigated'] = pseudo_890.origin.astype(str).isin(BAD_ORIGINS)
display(pseudo_890.style.format({
    'origin': '{:%Y-%m-%d}', 'actual_kg': '{:,.0f}',
    'absolute_error_kg_old': '{:,.0f}', 'absolute_error_kg_new': '{:,.0f}',
    'wape_old': '{:.2%}', 'wape_new': '{:.2%}', 'wape_change_pp': '{:+.2f}',
    'ratio_old': '{:.3f}', 'ratio_new': '{:.3f}',
}))

summary = pd.DataFrame({
    'scope': ['all 20 origins', 'three investigated origins', 'other 17 origins'],
    'wape_old': [
        pseudo_890.absolute_error_kg_old.sum() / pseudo_890.actual_kg.sum(),
        pseudo_890.loc[pseudo_890.investigated, 'absolute_error_kg_old'].sum()
            / pseudo_890.loc[pseudo_890.investigated, 'actual_kg'].sum(),
        pseudo_890.loc[~pseudo_890.investigated, 'absolute_error_kg_old'].sum()
            / pseudo_890.loc[~pseudo_890.investigated, 'actual_kg'].sum(),
    ],
    'wape_new': [
        pseudo_890.absolute_error_kg_new.sum() / pseudo_890.actual_kg.sum(),
        pseudo_890.loc[pseudo_890.investigated, 'absolute_error_kg_new'].sum()
            / pseudo_890.loc[pseudo_890.investigated, 'actual_kg'].sum(),
        pseudo_890.loc[~pseudo_890.investigated, 'absolute_error_kg_new'].sum()
            / pseudo_890.loc[~pseudo_890.investigated, 'actual_kg'].sum(),
    ],
})
summary['wape_change_pp'] = 100 * (summary.wape_new - summary.wape_old)
display(summary.style.format({
    'wape_old': '{:.2%}', 'wape_new': '{:.2%}', 'wape_change_pp': '{:+.2f}',
}))

,origin,actual_kg,absolute_error_kg_old,wape_old,ratio_old,absolute_error_kg_new,wape_new,ratio_new,wape_change_pp,investigated
0,2026-03-02,"147,760","32,219",21.81%,0.959,"33,659",22.78%,0.960,+0.97,False
1,2026-03-09,"128,642","28,687",22.30%,1.112,"28,769",22.36%,1.115,+0.06,False
2,2026-03-16,"127,959","24,772",19.36%,1.049,"24,896",19.46%,1.047,+0.10,False
3,2026-03-23,"115,017","25,695",22.34%,1.098,"24,269",21.10%,1.089,-1.24,False
4,2026-03-30,"178,734","46,591",26.07%,0.806,"44,353",24.82%,0.858,-1.25,True
5,2026-04-06,"106,989","20,355",19.03%,1.095,"22,264",20.81%,1.100,+1.78,False
6,2026-04-13,"124,003","43,879",35.39%,1.320,"46,639",37.61%,1.348,+2.23,False
7,2026-04-20,"123,267","30,928",25.09%,0.909,"29,250",23.73%,0.919,-1.36,False
8,2026-04-27,"151,230","47,315",31.29%,0.777,"49,183",32.52%,0.764,+1.23,True
9,2026-05-04,"131,903","31,579",23.94%,0.871,"31,600",23.96%,0.870,+0.02,False


,scope,wape_old,wape_new,wape_change_pp
0,all 20 origins,25.79%,25.89%,+0.10
1,three investigated origins,31.46%,31.03%,-0.43
2,other 17 origins,24.39%,24.61%,+0.23


## Daily view of the three bad weeks

The rows that mattered most in `05_04` are the pre-closure days
(2026-04-02, 2026-04-04, 2026-04-30, 2026-05-22, 2026-05-23). Ratios moving from
~0.6–0.8 toward 1.0 on exactly those days — without the other days degrading —
is the intended, attributable effect of the position feature.

In [5]:
bad_daily = con.execute(f'''
    WITH old_day AS (
        SELECT origin, period, SUM(actual) AS actual_kg, SUM(forecast) AS forecast_old,
               SUM(ABS(forecast - actual)) AS error_old
        FROM (SELECT ARTIKEL_ID, origin, period, SUM(actual) AS actual, SUM(forecast) AS forecast
              FROM forecasts_old
              WHERE sourcing_group = '{SOURCE_GROUP}' AND category_id = {CATEGORY_ID}
              GROUP BY 1, 2, 3)
        GROUP BY 1, 2
    ), new_day AS (
        SELECT origin, period, SUM(forecast) AS forecast_new,
               SUM(ABS(forecast - actual)) AS error_new
        FROM (SELECT ARTIKEL_ID, origin, period, SUM(actual) AS actual, SUM(forecast) AS forecast
              FROM forecasts_new
              WHERE sourcing_group = '{SOURCE_GROUP}' AND category_id = {CATEGORY_ID}
              GROUP BY 1, 2, 3)
        GROUP BY 1, 2
    )
    SELECT o.origin, o.period, o.actual_kg, o.forecast_old, n.forecast_new,
           o.forecast_old / NULLIF(o.actual_kg, 0) AS ratio_old,
           n.forecast_new / NULLIF(o.actual_kg, 0) AS ratio_new,
           o.error_old AS article_day_error_old, n.error_new AS article_day_error_new
    FROM old_day AS o
    INNER JOIN new_day AS n USING (origin, period)
    WHERE origin IN (SELECT UNNEST(?::DATE[]))
    ORDER BY origin, period
''', [BAD_ORIGINS]).fetchdf()
calendar = _holiday_calendar('2026-03-01', '2026-07-31')
calendar['period'] = pd.to_datetime(calendar.period)
bad_daily['period'] = pd.to_datetime(bad_daily.period)
bad_daily = bad_daily.merge(
    calendar[['period', 'event_name', 'holiday_event_window', 'days_to_nearest_event']],
    on='period', how='left', validate='many_to_one',
)
bad_daily.insert(2, 'weekday', bad_daily.period.dt.day_name().str[:3])
display(bad_daily.style.format({
    'origin': '{:%Y-%m-%d}', 'period': '{:%Y-%m-%d}',
    'actual_kg': '{:,.0f}', 'forecast_old': '{:,.0f}', 'forecast_new': '{:,.0f}',
    'ratio_old': '{:.3f}', 'ratio_new': '{:.3f}',
    'article_day_error_old': '{:,.0f}', 'article_day_error_new': '{:,.0f}',
}))

,origin,period,weekday,actual_kg,forecast_old,forecast_new,ratio_old,ratio_new,article_day_error_old,article_day_error_new,event_name,holiday_event_window,days_to_nearest_event
0,2026-03-30,2026-03-30,Mon,"26,235","20,823","21,205",0.794,0.808,"6,764","6,481",none,none,4
1,2026-03-30,2026-03-31,Tue,"27,252","21,685","23,226",0.796,0.852,"6,647","5,824",Karfreitag,before_holiday_1_3d,3
2,2026-03-30,2026-04-01,Wed,"32,538","27,773","30,435",0.854,0.935,"7,511","6,622",Karfreitag,before_holiday_1_3d,2
3,2026-03-30,2026-04-02,Thu,"42,414","34,906","37,386",0.823,0.881,"9,335","8,928",Karfreitag,before_holiday_1_3d,1
4,2026-03-30,2026-04-04,Sat,"50,295","38,858","41,079",0.773,0.817,"16,334","16,498",Karfreitag,after_holiday_1_3d,-1
5,2026-04-27,2026-04-27,Mon,"14,144","13,145","14,052",0.929,0.993,"2,423","2,285",none,none,4
6,2026-04-27,2026-04-28,Tue,"15,274","16,084","15,822",1.053,1.036,"3,015","2,719",Erster Mai,before_holiday_1_3d,3
7,2026-04-27,2026-04-29,Wed,"26,515","25,354","24,327",0.956,0.918,"4,985","5,348",Erster Mai,before_holiday_1_3d,2
8,2026-04-27,2026-04-30,Thu,"56,506","33,742","31,895",0.597,0.564,"25,030","26,842",Erster Mai,before_holiday_1_3d,1
9,2026-04-27,2026-05-02,Sat,"38,791","29,140","29,471",0.751,0.760,"11,862","11,989",Erster Mai,after_holiday_1_3d,-1


## Calendar contexts across all Pseudo 890 evaluation weeks

The `05_04` context view, now old versus new. The `day before public holiday`
context (three target dates) is where the fix should act; `week after Easter` is the
known **over**forecast context that this batch deliberately does not target.

In [6]:
school_ranges = [
    ('2025-04-07', '2025-04-19'), ('2025-05-30', '2025-05-30'),
    ('2025-06-10', '2025-06-10'), ('2025-07-03', '2025-08-13'),
    ('2025-10-13', '2025-10-25'), ('2025-12-22', '2026-01-05'),
    ('2026-02-02', '2026-02-03'), ('2026-03-23', '2026-04-07'),
    ('2026-05-15', '2026-05-15'), ('2026-05-26', '2026-05-26'),
    ('2026-07-02', '2026-08-12'),
]
tags = _holiday_calendar('2026-02-01', '2026-07-31').copy()
tags['period'] = pd.to_datetime(tags.period)
holiday_dates = set(tags.loc[tags.is_public_holiday, 'period'])
tags['day_before_public_holiday'] = tags.period.add(pd.Timedelta(days=1)).isin(holiday_dates)
tags['bridge_day'] = (
    (tags.period.dt.weekday.eq(4) & tags.period.sub(pd.Timedelta(days=1)).isin(holiday_dates))
    | (tags.period.dt.weekday.eq(0) & tags.period.add(pd.Timedelta(days=1)).isin(holiday_dates))
)
tags['school_holiday'] = False
for first_day, last_day in school_ranges:
    tags.loc[tags.period.between(first_day, last_day), 'school_holiday'] = True
easter_monday = pd.Timestamp('2026-04-06')
tags['week_after_easter'] = tags.period.between(
    easter_monday + pd.Timedelta(days=7), easter_monday + pd.Timedelta(days=13)
)
tags['calendar_context'] = np.select(
    [
        tags.is_public_holiday, tags.day_before_public_holiday, tags.bridge_day,
        tags.week_after_easter, tags.school_holiday,
    ],
    [
        'public holiday', 'day before public holiday', 'bridge day',
        'week after Easter', 'school holiday',
    ],
    default='ordinary',
)
con.register('calendar_context_tags', tags[['period', 'calendar_context']])

context_comparison = con.execute(f'''
    WITH old_day AS (
        SELECT period, SUM(actual) AS actual_kg, SUM(forecast) AS forecast_old,
               SUM(ABS(forecast - actual)) AS error_old
        FROM (SELECT ARTIKEL_ID, origin, period, SUM(actual) AS actual, SUM(forecast) AS forecast
              FROM forecasts_old
              WHERE sourcing_group = '{SOURCE_GROUP}' AND category_id = {CATEGORY_ID}
              GROUP BY 1, 2, 3)
        GROUP BY 1
    ), new_day AS (
        SELECT period, SUM(forecast) AS forecast_new,
               SUM(ABS(forecast - actual)) AS error_new
        FROM (SELECT ARTIKEL_ID, origin, period, SUM(actual) AS actual, SUM(forecast) AS forecast
              FROM forecasts_new
              WHERE sourcing_group = '{SOURCE_GROUP}' AND category_id = {CATEGORY_ID}
              GROUP BY 1, 2, 3)
        GROUP BY 1
    )
    SELECT t.calendar_context, COUNT(*) AS target_dates,
           SUM(o.actual_kg) AS actual_kg,
           SUM(o.error_old) / NULLIF(SUM(o.actual_kg), 0) AS wape_old,
           SUM(n.error_new) / NULLIF(SUM(o.actual_kg), 0) AS wape_new,
           SUM(o.forecast_old) / NULLIF(SUM(o.actual_kg), 0) AS ratio_old,
           SUM(n.forecast_new) / NULLIF(SUM(o.actual_kg), 0) AS ratio_new
    FROM old_day AS o
    INNER JOIN new_day AS n USING (period)
    INNER JOIN calendar_context_tags AS t USING (period)
    GROUP BY 1 ORDER BY actual_kg DESC
''').fetchdf()
display(context_comparison.style.format({
    'target_dates': '{:,.0f}', 'actual_kg': '{:,.0f}',
    'wape_old': '{:.2%}', 'wape_new': '{:.2%}',
    'ratio_old': '{:.3f}', 'ratio_new': '{:.3f}',
}))

,calendar_context,target_dates,actual_kg,wape_old,wape_new,ratio_old,ratio_new
0,ordinary,78,"1,708,984",25.80%,25.82%,0.986,0.992
1,school holiday,27,"600,593",22.43%,21.96%,0.941,0.961
2,day before public holiday,3,"135,858",32.03%,33.84%,0.716,0.713
3,week after Easter,6,"124,003",35.39%,37.61%,1.320,1.348
4,bridge day,1,"25,184",24.33%,23.86%,1.024,1.043


## Which stage moved?

The `05_04` decomposition on the three bad origins: occurrence calibration
(predicted minus actual positive-row rate) and the positive-quantity ratio
(predicted conditional quantity over actual, on rows with positive demand). The fix
routes position occurrence lift to the occurrence booster and position quantity lift
to the Gamma booster, so the quantity ratio should move most.

In [7]:
def stage_metrics(label):
    return con.execute(f'''
        SELECT origin,
               100 * (AVG(occurrence_probability) - AVG((actual > 0)::INTEGER))
                   AS occurrence_gap_pp,
               AVG(positive_quantity_forecast) FILTER (WHERE actual > 0)
                   / AVG(actual) FILTER (WHERE actual > 0) AS positive_mean_ratio
        FROM forecasts_{label}
        WHERE sourcing_group = '{SOURCE_GROUP}' AND category_id = {CATEGORY_ID}
              AND origin IN (SELECT UNNEST(?::DATE[]))
        GROUP BY origin ORDER BY origin
    ''', [BAD_ORIGINS]).fetchdf()

stage_comparison = stage_metrics('old').merge(
    stage_metrics('new'), on='origin', suffixes=('_old', '_new'), validate='one_to_one'
)
display(stage_comparison.style.format({
    'origin': '{:%Y-%m-%d}',
    'occurrence_gap_pp_old': '{:+.2f}', 'occurrence_gap_pp_new': '{:+.2f}',
    'positive_mean_ratio_old': '{:.3f}', 'positive_mean_ratio_new': '{:.3f}',
}))

,origin,occurrence_gap_pp_old,positive_mean_ratio_old,occurrence_gap_pp_new,positive_mean_ratio_new
0,2026-03-30,-2.36,0.829,-1.72,0.877
1,2026-04-27,-2.62,0.804,-2.56,0.790
2,2026-05-18,-0.69,0.759,-0.75,0.794


## Guardrail: the whole portfolio, not just Pseudo 890

Article-day WAPE and pooled bias (ratio − 1) per business segment and overall.
The change is acceptable only if Pseudo 890 improves without materially degrading
the other segments or pushing the overall bias away from zero.

In [8]:
def segment_metrics(label):
    return con.execute(f'''
        WITH article_day AS (
            SELECT sourcing_group || ' ' || category_id AS segment,
                   ARTIKEL_ID, origin, period,
                   SUM(actual) AS actual, SUM(forecast) AS forecast
            FROM forecasts_{label}
            GROUP BY 1, 2, 3, 4
        ), by_segment AS (
            SELECT segment, SUM(actual) AS actual_kg,
                   SUM(ABS(forecast - actual)) AS error_kg,
                   SUM(forecast) AS forecast_kg
            FROM article_day GROUP BY 1
        )
        SELECT segment, actual_kg, error_kg, forecast_kg FROM by_segment
        UNION ALL
        SELECT 'TOTAL', SUM(actual_kg), SUM(error_kg), SUM(forecast_kg) FROM by_segment
    ''').fetchdf()

portfolio = segment_metrics('old').merge(
    segment_metrics('new'), on='segment', suffixes=('_old', '_new'), validate='one_to_one'
)
portfolio['wape_old'] = portfolio.error_kg_old / portfolio.actual_kg_old
portfolio['wape_new'] = portfolio.error_kg_new / portfolio.actual_kg_new
portfolio['wape_change_pp'] = 100 * (portfolio.wape_new - portfolio.wape_old)
portfolio['bias_old'] = portfolio.forecast_kg_old / portfolio.actual_kg_old - 1
portfolio['bias_new'] = portfolio.forecast_kg_new / portfolio.actual_kg_new - 1
display(portfolio[[
    'segment', 'actual_kg_old', 'wape_old', 'wape_new', 'wape_change_pp',
    'bias_old', 'bias_new',
]].rename(columns={'actual_kg_old': 'actual_kg'}).style.format({
    'actual_kg': '{:,.0f}', 'wape_old': '{:.2%}', 'wape_new': '{:.2%}',
    'wape_change_pp': '{:+.2f}', 'bias_old': '{:+.2%}', 'bias_new': '{:+.2%}',
}))

,segment,actual_kg,wape_old,wape_new,wape_change_pp,bias_old,bias_new
0,FCM 900,"59,637",24.89%,25.64%,+0.76,+4.27%,+5.40%
1,FCM 890,"4,248",91.67%,92.44%,+0.77,+50.77%,+53.05%
2,Pseudo 890,"2,594,623",25.79%,25.89%,+0.10,-2.21%,-1.21%
3,Pseudo 900,"2,950",41.05%,41.26%,+0.20,+13.39%,+14.59%
4,TOTAL,"2,661,458",25.89%,26.01%,+0.11,-1.97%,-0.96%


## Did the model actually use the new features?

Gain share of the new position features in the refit blocks, alongside the
window-level pooled lifts they refine. Position values exist on a minority of rows
(event windows only), so even a modest global gain share is meaningful; the daily
tables above are the real effect measure.

In [9]:
new_importance = pd.read_csv(
    result_path(TWO_STAGE_MODEL_NAME, design, artifact='feature_importance')
)
lift_importance = (
    new_importance.loc[new_importance.feature.str.contains('lift')]
    .assign(rank=lambda frame: frame.groupby(['evaluation_origin', 'stage'])
            .gain_share.rank(ascending=False))
    .groupby(['stage', 'feature'])
    .agg(mean_gain_share=('gain_share', 'mean'),
         max_gain_share=('gain_share', 'max'),
         mean_rank=('rank', 'mean'))
    .reset_index()
    .sort_values(['stage', 'mean_gain_share'], ascending=[True, False])
)
display(lift_importance.style.format({
    'mean_gain_share': '{:.4%}', 'max_gain_share': '{:.4%}', 'mean_rank': '{:.1f}',
}))

,stage,feature,mean_gain_share,max_gain_share,mean_rank
3,occurrence,mean_action_lift_in_sourcing_group,0.0459%,0.0544%,1.0
2,occurrence,event_position_lift_occurrence,0.0284%,0.0342%,2.0
1,occurrence,event_lift_series,0.0174%,0.0202%,3.0
0,occurrence,event_lift_pooled_occurrence,0.0031%,0.0066%,4.0
5,positive_quantity,event_lift_series,0.3558%,0.4307%,1.0
7,positive_quantity,mean_action_lift_in_sourcing_group,0.2339%,0.2756%,2.0
6,positive_quantity,event_position_lift_quantity,0.0353%,0.0839%,3.4
4,positive_quantity,event_lift_pooled_quantity,0.0255%,0.0448%,3.6


## Conclusions

**Bias — the primary goal — moved the right way.** Pooled bias improved from
−1.97% to −0.96% overall and from −2.21% to −1.21% for Pseudo 890, while overall
article-day WAPE stayed flat (25.89% → 26.01%, +0.11 pp, within refit noise). The
model now gives back part of the systematic pre-holiday underforecast without
buying that bias reduction with accuracy elsewhere.

**Two of the three investigated weeks improved, and they improved on exactly the
days the feature targets.**

- **2026-03-30 (Karfreitag/Easter):** WAPE 26.07% → 24.82%, weekly ratio
  0.806 → 0.858. The pre-closure days moved most: Wednesday 0.854 → 0.935,
  Thursday 0.823 → 0.881, Tuesday 0.796 → 0.852, Easter Saturday 0.773 → 0.817.
- **2026-05-18 (Pfingstmontag):** WAPE 36.77% → 35.78%, ratio 0.757 → 0.792;
  Friday 0.656 → 0.703 and Saturday 0.653 → 0.676.
- **2026-04-27 (Erster Mai) did not improve** (WAPE 31.29% → 32.52%, Thursday
  2026-04-30 ratio 0.597 → 0.564) even though the feature carried its largest
  signal there (position quantity lift 2.82, total 4.32).

**Why the Erster Mai day resisted the fix — a training-coverage ceiling.** The
≥2-distinct-dates support gate needs two historical years. During the 2025
training origins most events had only their 2024 occurrence in history, so just
13.5% of event-window training rows carry a position value, and the largest
position quantity lift the boosters ever saw in training is **1.64**. Karfreitag
and Pfingstmontag evaluation values (≈1.2–1.7) lie inside that trained range and
were acted on; the Erster Mai value of 2.82 lies far outside it, and gradient
boosted trees cannot extrapolate beyond their largest learned split threshold —
the 2.82 falls into the same leaf as ≈1.6. The feature is right; the model has
not yet seen enough rows in that value region to use it fully.

**The boosters do use the new features.** Among the lift features,
`event_position_lift_occurrence` ranks second in the occurrence stage (ahead of
both `event_lift_series` and the window-level pooled occurrence lift), and
`event_position_lift_quantity` ranks above the window-level quantity lift in the
Gamma stage. Absolute gain shares stay small because position values exist on a
minority of rows, which is expected; the daily tables above are the effect
measure that matters.

**What this batch deliberately did not touch, and what to try next (one batch
each):**

1. **Extend the trained value range of the position lift.** Replace the hard
   ≥2-dates gate with shrinkage toward the window-level lift (precision-weighted
   by date count). The 2025 training rows would then carry non-missing position
   values estimated from 2024 alone — including the ~3.5× Erster Mai peak — and
   the quantity booster could learn split thresholds in the high-lift region that
   2026-04-30 needs. This is the single most promising follow-up.
2. **Week-after-Easter overforecast** (2026-04-13, ratio 1.320 → 1.348, the worst
   remaining origin by WAPE): a separate mechanism — trailing windows
   (`rolling_24_mean` and relatives) inflated by the Easter run-up echo into the
   quiet post-holiday week. A candidate feature is a rolling mean computed over
   non-event-window days only, or a post-event-week flag with its own pooled lift.
3. **Reopen days after holiday closures** (Saturday 2026-05-02, 0.751 → 0.760):
   position lift exists here but historical post-May-1 offsets fell on different
   weekdays across years, weakening the match; a closure-relative key (first
   trading day after a holiday block) would pool these cleanly.

**Scope note.** Only the two-stage model was refit; the direct L2/Tweedie/weekly
results on disk still come from the previous feature generation. The comparison
against the backed-up run shares data, protocol, and hyperparameters, so the
differences shown here are attributable to the three added feature columns plus
LightGBM's refit stochasticity (bagging); single-origin WAPE changes of ±1 pp on
untargeted weeks are within that noise level.